**11/28/2025**

## GBM Hybrid Framework Encoding

Datasets: Methylation27k (DNA methylation), Phenotype (Clinical Data + ground truths), HiSeqV2 (mRNA seq)


https://xenabrowser.net/datapages/?cohort=GDC%20TCGA%20Glioblastoma%20(GBM)&removeHub=https%3A%2F%2Fxena.treehouse.gi.ucsc.edu%3A443


### Data Preprocessing and Feature Engineering Protocol

This cell performs the complete end-to-end data preparation:

1.  **Data Loading and Cleaning**: Loads RNA, Methylation, and Clinical data, cleans patient barcodes, and ensures strict sample alignment and deduplication across all modalities.
2.  **Feature Compression**: Selects the top variable genomic features (RNA and Meth) and compresses them into 40 semantic pathway groups using mean aggregation. Selects the top 20 sparse clinical features.
3.  **Spike Train Encoding**:
    * **Transciptomics and Epigenetics (RNA Seq, Methylation)**: Encodes continuous compressed features using **Time-to-First-Spike (TTFS)** encoding over T=100 time steps.
    * **Clinical (Phenotype)**: Encodes sparse one-hot clinical features using **Temporal** encoding over T=100 time steps.
4.  **Final Split and Save**: Removes rare classes, performs a robust final index alignment, and splits the data into 80% Cross-Validation (CV) and 20% final Test sets, saving the resulting sparse spike train matrices and labels to CSV files.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import warnings

# Suppress minor warnings for clean output
warnings.filterwarnings('ignore')

# --- PROTOCOL PARAMETERS ---
T = 100
N_SEMANTIC_FEATURES = 60

# --- PROTOCOL DESIGN 2: HYBRID ENCODING FUNCTIONS ---
def ttfs_encode_sparse(data, T=100):
    """Time-to-First-Spike (TTFS) Encoding for Genomics (Modality 01)."""
    min_val = data.min().min()
    max_val = data.max().max()
    normalized_data = (data.values - min_val) / (max_val - min_val + 1e-6)
    time_indices = np.floor((1 - normalized_data) * (T - 1)).astype(int)
    N_samples, N_features = data.shape
    col_names = [f"01.{c}.G.t{t}" for c in data.columns for t in range(T)]
    spike_trains = pd.DataFrame(0, index=data.index, columns=col_names, dtype=np.int8)
    for i in range(N_samples):
        for j in range(N_features):
            time_index = time_indices[i, j]
            col_name = f"01.{data.columns[j]}.G.t{time_index}"
            spike_trains.loc[data.index[i], col_name] = 1
    spike_trains = spike_trains.loc[:, spike_trains.sum() > 0]
    return spike_trains

def temporal_encode_sparse(data, T=100):
    """Temporal Encoding for Clinical (Modality 03)."""
    N_features = data.shape[1]
    col_names = [f"03.{c}.C.t{t}" for c in data.columns for t in range(T)]
    spike_trains = pd.DataFrame(0, index=data.index, columns=col_names, dtype=np.int8)
    for j in range(N_features):
        time_step = (j % T)
        samples_with_feature = data.index[data.iloc[:, j] == 1]
        col_name = f"03.{data.columns[j]}.C.t{time_step}"
        if not samples_with_feature.empty:
            spike_trains.loc[samples_with_feature, col_name] = 1
    spike_trains = spike_trains.loc[:, spike_trains.sum() > 0]
    return spike_trains


# --- DATA LOADING AND ALIGNMENT (CRITICAL DEDUPLICATION INCLUDED) ---
try:
    rna = pd.read_csv("HiSeqV2", sep="\t", index_col=0)
    meth = pd.read_csv("HumanMethylation27", sep="\t", index_col=0)
    clin = pd.read_csv("TCGA.GBM.sampleMap_GBM_clinicalMatrix", sep="\t", index_col=0)
except FileNotFoundError:
    print("FATAL ERROR: Raw data files not found. Please ensure HiSeqV2, HumanMethylation27, and TCGA.GBM.sampleMap_GBM_clinicalMatrix are loaded.")
    raise

# [CLEANING AND ALIGNMENT STEPS...]
def clean_barcode(x):
    return x.str[:12].str.upper()

rna.columns = clean_barcode(rna.columns)
meth.columns = clean_barcode(meth.columns)
clin.index = clean_barcode(clin.index)

label_candidates = ["GeneExp_Subtype", "Subtype_Classification", "TCGA_Subtype", "PAM50", "GeneExpression.Subtype", "Subtype"]
subtype_col = next((c for c in clin.columns if c in label_candidates), None)
labels = clin.pop(subtype_col).astype(str)

common_samples = sorted(list(set(rna.columns) & set(meth.columns) & set(clin.index) & set(labels.index)))

rna = rna[common_samples].T.loc[:, rna.T.var() > 0].fillna(rna.T.mean()).astype(float)
meth = meth[common_samples].T.loc[:, meth.T.var() > 0].fillna(meth.T.mean()).astype(float)
clin = clin.loc[common_samples]
labels = labels.loc[common_samples]

common_samples = sorted(list(set(rna.index) & set(meth.index) & set(clin.index) & set(labels.index)))
rna = rna.loc[common_samples]
meth = meth.loc[common_samples]
clin = clin.loc[common_samples]
labels = labels.loc[common_samples]

# Final definitive deduplication across all modalities
rna = rna[~rna.index.duplicated(keep='first')]
meth = meth[~meth.index.duplicated(keep='first')]
clin = clin[~clin.index.duplicated(keep='first')]
labels = labels.loc[rna.index]
clin = clin.loc[rna.index]

print(f"Total Aligned Samples (N): {len(rna.index)}")
print(f"Initial Genomics Features (RNA + Meth): {rna.shape[1] + meth.shape[1]}")


# --- PROTOCOL DESIGN 1: FUNCTIONAL PATHWAY COMPRESSION ---
N_SEMANTIC_GROUPS = 40
N_RNA_GROUPS = N_SEMANTIC_GROUPS // 2
N_METH_GROUPS = N_SEMANTIC_GROUPS // 2
rna_top_features = rna.var(axis=0).sort_values(ascending=False).head(2000).index
meth_top_features = meth.var(axis=0).sort_values(ascending=False).head(2000).index
np.random.seed(42)
rna_map = {feature: f"RNA_Pathway_{i % N_RNA_GROUPS}" for i, feature in enumerate(rna_top_features)}
meth_map = {feature: f"METH_Pathway_{i % N_METH_GROUPS}" for i, feature in enumerate(meth_top_features)}
all_genomics_data = pd.concat([rna[rna_top_features], meth[meth_top_features]], axis=1)
combined_map = {**rna_map, **meth_map}
mapped_data = all_genomics_data.T.loc[all_genomics_data.columns.intersection(combined_map.keys())]
genomics_compressed = mapped_data.groupby(combined_map).mean().T

# Genomics Encoding
genomics_standardized = pd.DataFrame(
    StandardScaler().fit_transform(genomics_compressed),
    index=genomics_compressed.index,
    columns=genomics_compressed.columns
)
genomics_spikes = ttfs_encode_sparse(genomics_standardized, T=T)

# Clinical Encoding
clin_clean = clin.drop(columns=clin.columns[clin.nunique() > 10].tolist()).dropna(axis=1, how='all')
clin_ohe = pd.get_dummies(clin_clean.astype(str), prefix=clin_clean.columns, dummy_na=False).astype(int)
N_CLIN_FEATURES = N_SEMANTIC_FEATURES - genomics_compressed.shape[1]
top_feature_names = clin_ohe.sum().sort_values(ascending=False).head(N_CLIN_FEATURES).index
clin_ohe_subset = clin_ohe[top_feature_names]
clin_ohe_subset = clin_ohe_subset.reindex(genomics_compressed.index).fillna(0).astype(int)
clinical_spikes = temporal_encode_sparse(clin_ohe_subset, T=T)

# --- FINAL PROTOCOL ASSEMBLY (INDEX ALIGNMENT) ---
X_protocol = pd.concat([genomics_spikes, clinical_spikes], axis=1)

# CRITICAL FIX 1: Ensure X_protocol index is strictly unique
# This step is critical as encoding can sometimes re-introduce or fail to clear duplicates if not done carefully.
X_protocol = X_protocol[~X_protocol.index.duplicated(keep='first')]
unique_final_index = X_protocol.index

label_encoder = LabelEncoder()
aligned_labels = labels.loc[unique_final_index]
y_protocol = pd.Series(label_encoder.fit_transform(aligned_labels),
                       index=aligned_labels.index,
                       name="label")

# CRITICAL FIX 2: Final explicit index intersection to guarantee alignment before rare-class removal
final_common_index = X_protocol.index.intersection(y_protocol.index)
X_protocol = X_protocol.loc[final_common_index]
y_protocol = y_protocol.loc[final_common_index]


print("\n--- Protocol Final Output ---")
print(f"Total Protocol Input Neurons (D_protocol): {X_protocol.shape[1] / T}")
print(f"Total Spike Train Columns (D_protocol x T): {X_protocol.shape[1]}")


# --- DATA SPLITTING AND SAVING (FINAL ROBUST FIXES) ---
min_samples_per_class = 2
counts = y_protocol.value_counts()
rare_classes = counts[counts < min_samples_per_class].index.tolist()

if rare_classes:
    print(f"\nRemoving samples corresponding to rare classes: {rare_classes}")
    mask = ~y_protocol.isin(rare_classes)

    # Fix for previous IndexError: Slice by index names (safe)
    keep_indices = mask[mask].index

    X_protocol = X_protocol.loc[keep_indices]
    y_protocol = y_protocol.loc[keep_indices]


# *** FIX FOR CURRENT ValueError ***
# 1. Force uniqueness one last time in case the slicing step above failed to preserve it perfectly.
X_protocol = X_protocol[~X_protocol.index.duplicated(keep='first')]
y_protocol = y_protocol[~y_protocol.index.duplicated(keep='first')]

# 2. Re-intersect and align indices using simple reindex (or loc),
# which is safe now that indices are guaranteed unique.
final_split_index = X_protocol.index.intersection(y_protocol.index)

# Use .loc slicing instead of .reindex() to avoid potential issues if reindex is somehow failing on unique indices.
X_protocol = X_protocol.loc[final_split_index]
y_protocol = y_protocol.loc[final_split_index]


# Perform the 80/20 stratified split.
X_cv, X_test, y_cv, y_test = train_test_split(
    X_protocol, y_protocol,
    test_size=0.20,
    stratify=y_protocol,
    random_state=42
)

print("\n--- Final Data Split (Protocol Ready) ---")
print("Cross-Validation Set (80% for K-Fold):", len(X_cv))
print("Final Test Set (20% Holdout):", len(X_test))

# SAVE FINAL SPARSE FILES
X_cv.to_csv("X_cv_protocol_sparse.csv")
y_cv.to_csv("y_cv_protocol_sparse.csv")
X_test.to_csv("X_test_final_protocol_sparse.csv")
y_test.to_csv("y_test_final_protocol_sparse.csv")

print("\nSaved Protocol-Ready Files (X_cv_protocol_sparse.csv, etc.)")

Total Aligned Samples (N): 80
Initial Genomics Features (RNA + Meth): 45009

--- Protocol Final Output ---
Total Protocol Input Neurons (D_protocol): 17.79
Total Spike Train Columns (D_protocol x T): 1779

Removing samples corresponding to rare classes: [4]

--- Final Data Split (Protocol Ready) ---
Cross-Validation Set (80% for K-Fold): 63
Final Test Set (20% Holdout): 16

Saved Protocol-Ready Files (X_cv_protocol_sparse.csv, etc.)


Export .csv files:

| File                   | Description          |
| ---------------------- | -------------------- |
| `X_cv_protocol_sparse.csv`          | cross-validation features    |
| `X_cv_protocol_sparse.csv`          | cross-validation features    |
| `X_test_final_protocol_sparse.csv`            | final test fetaures     |
| `y_test_final_protocol_sparse.csv`           | final test labels     |

Will manually save to Google Drive (Folder = `Multimodal_SNN_Protocol_Project`) and Github


In [ ]:
from google.colab import files

# List of files saved by the preprocessing script
files_to_download = [
    "X_cv_protocol_sparse.csv",
    "y_cv_protocol_sparse.csv",
    "X_test_final_protocol_sparse.csv",
    "y_test_final_protocol_sparse.csv"
]

for filename in files_to_download:
    try:
        files.download(filename)
        print(f"Initiated download for {filename}")
    except FileNotFoundError:
        print(f"Error: {filename} not found. Ensure the preprocessing cell ran successfully.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Initiated download for X_cv_protocol_sparse.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Initiated download for X_test_final_protocol_sparse.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Initiated download for y_test_final_protocol_sparse.csv


##Research Plan Amendments

*Due to an limited amount of GBM data, in order to better evaluate the contributions of the protocol in terms of model performance, scalability, and efficiency, the research plan has undergone a few changes.*

This project will now be split into two phases; Phase 1 and Phase 2.

- **Phase 1**: Small scale implementations using **TCGA-GBM** data for the purposes of protocol prototyping and evaulating performance with small scale data and small sample sizes (overfitting).

- **Phase 2**: Full scale implementations using data from the **TCGA-PANCAN** data atlas. This phase will be the main source of data for the project and will focus on model performance and efficiency.


Experimentation for Phase 2 will commence once approval has been recieved.
The updated research plan can be accessed through the project Google Drive folder (`Multimodal SNN Protocol Project`)

